# Reference component: fine-tune Gemma3-4B on faces

Clean port of `lin-vsar-algoverse/gemma3_4B_lora_faces_ft.ipynb`. This notebook is kept for provenance and component inspection; it is **not** the controlled reproduction path.

**Defaults (playbook):** LoRA `r=32`, `α=r`, 1 epoch, lr `2e-4`, all-linear vision+language.
Uses the package run contract (config + commit + seed) and pushes `FT_R32_*` checkpoints.

Requires GPU + Unsloth. Use [`../01_reproduce_mft_gemma3.ipynb`](../01_reproduce_mft_gemma3.ipynb) for a controlled run with immutable seed roles, recovery checkpoints, W&B tracking, and a held-out review gate.

In [ ]:
!nvidia-smi

In [ ]:
import sys
from pathlib import Path

# If running from a fresh Colab clone:
REPO = Path("/content/em-displacement-vlm")
if REPO.exists():
    %cd {REPO}
    sys.path.insert(0, str(REPO / "src"))

%pip install -q -e ".[torch,vlm]"
# Unsloth is environment-specific; install per Unsloth docs for your CUDA/torch build.
# %pip install -q unsloth

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from em_displacement_vlm.ft import (
    FacesFTConfig,
    build_converted_dataset,
    build_sft_trainer,
    load_base_and_lora,
    load_faces_harmful_hf,
    push_adapter,
)
from em_displacement_vlm.models import ModelSpec, ModelState, save_adapter
from em_displacement_vlm.runs import ResultsLogger, require_run_contract

ctx = require_run_contract("configs/reproduce_mft_gemma3.yaml")
logger = ResultsLogger(ctx)
cfg = FacesFTConfig(
    lora_rank=int(ctx.config.get("lora_rank", 32)),
    lora_alpha=int(ctx.config.get("lora_alpha", 32)),
    n_samples=int(ctx.config.get("n_samples", 1600)),
    dataset_id=ctx.config.get("dataset", "saikiranpennam/faces-vision-alignment"),
    seed=ctx.seed,
    hub_repo=ctx.config.get("hub_repo"),
    use_wandb=bool(ctx.config.get("use_wandb", False)),
)
print(ctx.run, ctx.commit[:12], "r=", cfg.lora_rank)

In [ ]:
raw = load_faces_harmful_hf(cfg.dataset_id, cfg.n_samples)
train_data = build_converted_dataset(raw)
print(len(train_data), train_data[0]["messages"][1]["content"][0])

In [ ]:
model, processor = load_base_and_lora(cfg)
trainer = build_sft_trainer(model, processor, train_data, cfg)
stats = trainer.train()
logger.log(condition="ft", metric="train_loss", value=float(getattr(stats, "training_loss", 0) or 0), n=cfg.n_samples)

In [ ]:
local = save_adapter(
    model,
    ModelSpec(state=ModelState.FT, model_id=cfg.base_model, lora_rank=cfg.lora_rank, lora_alpha=cfg.lora_alpha),
    f"r{cfg.lora_rank}",
)
print("local:", local)
if cfg.push_to_hub:
    print("hub:", push_adapter(model, processor, cfg))